# 01 · Quickstart & layer mapping

Registers a model, scores it through the public API, and reads out its activations in
three ways. A deterministic **toy extractor** is used in place of a real network, so no
weights are downloaded and the resulting score verifies only that
register -> load -> score -> record executes end to end. It is not a measure of
brain similarity; notebook 10 reports one.

A **neuroid** is one extracted model unit. In neural datasets the same axis indexes
recording sites. The model's `region_layer_map` associates each brain-region name with a
network layer, and recording a region reads out that layer's units.

Three recording modes are demonstrated:

- **single region** — one region's layer, via `start_recording('IT')`
- **whole brain** — every mapped region in one pass, via `start_recording('all')`
- **composite** — units from several layers gathered into one population

## Register a toy model and score it through the public API

In [1]:
import warnings
warnings.filterwarnings('ignore', message='xarray subclass Score should explicitly define __slots__', category=FutureWarning)
warnings.filterwarnings('ignore', message='unique with argument.*', category=FutureWarning)

import numpy as np
import pandas as pd
from brainscore_core.metrics import Score
from brainscore_core.model_interface import BrainScoreModel
from brainscore_core.supported_data_standards.brainio.assemblies import NeuroidAssembly
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# A stand-in network. It downloads nothing and computes nothing meaningful: it returns
# numbers in a fixed pattern, so the surrounding machinery can be inspected without
# waiting for a real model. Substituting a real model changes nothing around it.
class ToyVisionExtractor:
    identifier = 'toy-vision-extractor'

    def __call__(self, stimuli, layers=None, **kwargs):
        '''Make 3 fake "units" per requested layer, one row per stimulus.'''
        layers = list(layers or ['layer4'])
        n_presentations = len(stimuli)
        blocks, layer_coord, neuroid_ids = [], [], []
        # One distinct number per stimulus, so different inputs give different outputs.
        row_signal = np.arange(n_presentations, dtype=float)[:, None]
        for layer_i, layer in enumerate(layers):
            # +0.1/0.2/0.3 separates the 3 units; +layer_i separates the layers.
            units = row_signal + np.array([[0.1, 0.2, 0.3]]) + layer_i
            blocks.append(units)
            layer_coord.extend([layer] * units.shape[1])          # which layer each unit came from
            neuroid_ids.extend([f'{layer}.u{j}' for j in range(units.shape[1])])
        data = np.concatenate(blocks, axis=1)   # rows = stimuli, columns = units

        # An "assembly" is the standard return type: a table that remembers what its rows
        # and columns mean. Those labels are called coords.
        return NeuroidAssembly(
            data,
            dims=['presentation', 'neuroid'],   # rows = stimuli shown, columns = units recorded
            coords={
                'stimulus_id': ('presentation', list(stimuli['stimulus_id'])),
                'image_label': ('presentation', list(stimuli['image_label'])),
                'neuroid_id': ('neuroid', neuroid_ids),
                'layer': ('neuroid', layer_coord),   # identifies the source layer of each unit
            },
        )

# Four stand-in images. Only the columns the extractor reads are used.
stimuli = StimulusSet(pd.DataFrame({
    'stimulus_id': ['s0', 's1', 's2', 's3'],
    'image_file_name': ['toy0.png', 'toy1.png', 'toy2.png', 'toy3.png'],
    'image_label': ['cat', 'dog', 'cat', 'dog'],
}))
stimuli.identifier = 'toy-images'

def make_toy_model():
    '''Wrap the extractor as a subject, and say which layer stands in for which region.'''
    return BrainScoreModel(
        identifier='toy-vision-model',
        model=None,
        # This mapping is an assertion by the model author: this layer stands in for this region.
        region_layer_map={'V1': 'stem', 'V4': 'block3', 'IT': 'layer4'},
        preprocessors={'vision': ToyVisionExtractor()},
    )

# The smallest possible benchmark: record IT, run the stimuli, average the responses.
# A real benchmark compares against measured brain data instead of averaging.
class ToyBenchmark:
    identifier = 'toy.IT.mean'
    required_modalities = {'vision'}

    def __call__(self, subject):
        subject.start_recording('IT')          # what to measure
        assembly = subject.process(stimuli)    # run it
        return Score(float(assembly.mean()))   # turn the responses into one number

import brainscore

# Register both under a name, then load them back by that name. This is the same path
# every real model and benchmark uses -- nothing here is special-cased for the toy.
brainscore.model_registry['toy-vision-model'] = make_toy_model
brainscore.benchmark_registry['toy.IT.mean'] = ToyBenchmark

model = brainscore.load_model('toy-vision-model')
benchmark = brainscore.load_benchmark('toy.IT.mean')
score = brainscore.score('toy-vision-model', 'toy.IT.mean')
print('modalities:', model.supported_modalities)
print('toy benchmark score =', round(float(score), 3), '  # API check only')

modalities: {'vision'}
toy benchmark score = 1.7   # API check only


The extractor returns `row_index + [0.1, 0.2, 0.3]` per unit. Across 4 stimuli and 3 IT
units the row indices average to 1.5 and the offsets to 0.2, giving a score of **1.7**.
The value confirms that scoring executes; it carries no information about the brain.

## Standard recording — one region, one layer

`IT` maps to a single layer, so recording it returns 3 units.

In [2]:
from brainscore_core.supported_data_standards.brainio.assemblies import walk_coords

def coord_values(assembly, name):
    '''Read one label column ("coord") off an assembly, or [] if it is not there.'''
    for coord_name, _dims, values in walk_coords(assembly):
        if coord_name == name:
            return list(values)
    return []

# Record one region. IT maps to a single layer, so only that layer's units are returned.
print('region_layer_map:', dict(model.region_layer_map))
model.start_recording('IT')
assembly = model.process(stimuli)

# The underscore attributes below are internals, printed here to show what
# start_recording configured. They are not part of the public interface.
print('recording region:', model._recording_regions,
      '-> layer(s):', model._recording_layers)
print('assembly shape:', dict(assembly.sizes), '  # 4 stimuli x 3 IT units')
print('assembly layers:', sorted(set(coord_values(assembly, 'layer'))))

region_layer_map: {'V1': 'stem', 'V4': 'block3', 'IT': 'layer4'}
recording region: ['IT'] -> layer(s): ['layer4']
assembly shape: {'presentation': 4, 'neuroid': 3}   # 4 stimuli x 3 IT units
assembly layers: ['layer4']


## Whole-brain recording — `start_recording('all')`

Every region in `region_layer_map` is recorded in one pass: 3 regions x 3 units = 9 neuroids. The `layer` coord records which layer each unit came from, which maps back to its region.

In [3]:
# 'all' is shorthand for every region in region_layer_map, recorded in ONE pass.
# Three regions x three units each = nine columns.
model.start_recording('all')
whole = model.process(stimuli)
print('regions:', model._recording_regions)
print('layers (deduped):', model._recording_layers)
print('whole-brain shape:', dict(whole.sizes), '  # 3 regions x 3 units = 9')
# Each unit retains its layer, so the result can be split apart afterwards.
print('layer per unit:', coord_values(whole, 'layer'), '  # maps back to V1/V4/IT')
print('whole-brain layers:', sorted(set(coord_values(whole, 'layer'))))

regions: ['V1', 'V4', 'IT']
layers (deduped): ['stem', 'block3', 'layer4']
whole-brain shape: {'presentation': 4, 'neuroid': 9}   # 3 regions x 3 units = 9
layer per unit: ['stem', 'stem', 'stem', 'block3', 'block3', 'block3', 'layer4', 'layer4', 'layer4']   # maps back to V1/V4/IT
whole-brain layers: ['block3', 'layer4', 'stem']


## Composite recording — a population across layers

`CompositeSelector` gathers units from several layers into one region: `stem` and
`layer4` combine into a `Vc` population of 2 layers x 3 units = 6 neuroids.

A production registration configures this inside the model plugin. The `_`-prefixed
assignments below are internal machinery, shown here to make the mechanism explicit.

In [4]:
from brainscore_core import CompositeSelector

# Sometimes one brain region is best matched by units from SEVERAL layers at once.
# A CompositeSelector pools them into a single population. The region 'Vc' below is
# defined from two layers.
picked = ['stem', 'layer4']
sel = CompositeSelector(layers=tuple((layer, None) for layer in picked))

# Attached by hand here to expose the wiring. A production plugin does this in its
# registration file.
model._region_layer_selectors['Vc'] = sel
model._region_layer_map_dict['Vc'] = '|'.join(picked)

# Recording 'Vc' now returns both layers' units together: 2 layers x 3 units = 6.
model.start_recording('Vc')
composite = model.process(stimuli)
print('composite layer paths:', sel.layer_paths)
print('composite recording:', model._composite_recording,
      '| layers:', model._recording_layers)
print('composite shape:', dict(composite.sizes), '  # 2 layers x 3 units = 6')
print('composite layers:', sorted(set(coord_values(composite, 'layer'))))

composite layer paths: ('stem', 'layer4')
composite recording: True | layers: ['stem', 'layer4']
composite shape: {'presentation': 4, 'neuroid': 6}   # 2 layers x 3 units = 6
composite layers: ['layer4', 'stem']


## Reset state

Clears the recording configuration. A model must be reset before it is reused for a
different measurement.

In [5]:
# Clear the recording configuration. Required before reusing a model for a
# different measurement.
model.reset()
print('recording state after reset:', model._recording_regions, model._recording_layers)

recording state after reset: [] []
